# Scenario: Streaming Responses

The Responses API can return tokens incrementally via streaming. This notebook validates:

- Chunking logic: the stream yields multiple events/chunks.
- Full message capture: iterating the stream and concatenating content yields the complete response.
- Response type and status: streamed response completes successfully and matches non-streamed content semantics.

Configuration: `BASE_URL` / `INFERENCE_MODEL`. Server assumed running (e.g. `http://localhost:8321`).


## Setup

Load base URL and model from environment; create the OGX client.


In [ ]:
import os

from openai import OpenAI

base_url = os.environ.get("BASE_URL")
model = os.environ.get("INFERENCE_MODEL")

assert base_url, "BASE_URL must be set"
assert model, "INFERENCE_MODEL must be set"

openai_base_url = base_url.rstrip("/")
openai_base_url = (
    openai_base_url if openai_base_url.endswith("/v1") else openai_base_url + "/v1"
)
client = OpenAI(api_key="no-key-needed", base_url=openai_base_url)

## Stream iteration and full message capture

Call `responses.create(..., stream=True)` and iterate over the stream. Collect all text deltas to form the full message. Assert we receive multiple chunks and that the concatenated result is non-empty and coherent.


In [ ]:
prompt = "What is the capital of France? Answer in one short sentence."
stream = client.responses.create(
    model=model,
    input=prompt,
    stream=True,
)

chunks = []
full_text = ""
for chunk in stream:
    chunks.append(chunk)
    # Text deltas arrive as OutputTextDelta chunks with a .delta attribute
    delta = getattr(chunk, "delta", None)
    if isinstance(delta, str):
        full_text += delta

assert len(chunks) >= 1, "Expected at least one streamed chunk"
assert full_text.strip(), "Expected non-empty full message from stream"
assert "Paris" in full_text, (
    f"Expected Paris in streamed response, got: {full_text[:200]}"
)